# Which Clusters Change the Most from Init to Trained MFA?

Companion to `kmeans_init_vs_mfa_comparison.ipynb`, focused on one question:
**per cluster, how many samples change membership between the k-means init
partition and the trained MFA partition?**

Inputs (same token stream, same K, identity cluster correspondence):

- **k-means (init)**: `kmeans_centroid_assignments.pt` — nearest Euclidean
  centroid per token.
- **MFA (trained)**: `mfa_model_assignments.pt` — argmax-responsibility
  component per token.

For each cluster `k` we count, from the K×K contingency matrix:

- `left_k`   = samples in k-means cluster k that end up elsewhere under MFA
- `joined_k` = samples in MFA cluster k that came from another init cluster
- `churn_k`  = `left_k + joined_k` — the **membership-change count**
  (size of the symmetric difference `km_k Δ mfa_k`)
- `net_delta_k` = `mfa_size_k − km_size_k` — the net size change

and plot the distributions of these deltas, raw and normalized by cluster size.


## 1. Setup and Artifact Validation

In [1]:
from __future__ import annotations

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

try:
    import plotly.express as px
    import plotly.graph_objects as go
except Exception as exc:
    px = None
    go = None
    print(f"Plotly unavailable: {exc}")


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "src/dalg").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


REPO = find_repo_root()

# ---------------------------------------------------------------- parameters
LAYER = 5
K = 1000
Q = 10
EPOCHS = 1
# ------------------------------------------------- paths built from parameters
MFA_RUN = REPO / f"dalg-cache/pile_gemma2b_models/layer{LAYER:02d}_{K}_{Q}_component_sharded_mfa"
MFA_ASSIGN_PATH = MFA_RUN / "mfa_model_assignments.pt"
MFA_INIT_CENTROIDS = MFA_RUN / "centroids.pt"

CENTROIDS_DIR = REPO / f"dalg-cache/pile_gemma2b_models/centroids/k{K}_L{LAYER:02d}"
KMEANS_CENTROIDS = CENTROIDS_DIR / "centroids.pt"
KMEANS_ASSIGN = CENTROIDS_DIR / "kmeans_centroid_assignments.pt"

# ------------------------------------------------------------- plot constants
COLOR_KMEANS = "#2a78d6"  # blue  — k-means init partition
COLOR_MFA = "#1baf7a"     # aqua  — trained MFA partition
COLOR_NEUTRAL = "#52514e" # gray  — derived quantities (deltas)
PLOT_TEMPLATE = "plotly_white"

PLOTS_DIR = REPO / "notebooks/plots"
PLOTS_HTML_DIR = PLOTS_DIR / "html"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_HTML_DIR.mkdir(parents=True, exist_ok=True)
PLOT_TAG = f"L{LAYER:02d}_K{K}_q{Q}_ep{EPOCHS}_membership"


def save_fig(fig, name: str) -> None:
    fig.update_layout(title_font_size=13, title_x=0.5, margin=dict(t=48))
    path = PLOTS_DIR / f"{name}_{PLOT_TAG}.pdf"
    try:
        fig.write_image(str(path), width=1000, height=550)
    except Exception as exc:
        path = PLOTS_HTML_DIR / f"{name}_{PLOT_TAG}.html"
        fig.write_html(str(path), include_plotlyjs="cdn")
        print(f"PDF export failed ({exc}); saved {path.name} instead")


artifacts = pd.DataFrame(
    [
        {"artifact": "kmeans assignments", "path": str(KMEANS_ASSIGN), "exists": KMEANS_ASSIGN.exists()},
        {"artifact": "mfa assignments", "path": str(MFA_ASSIGN_PATH), "exists": MFA_ASSIGN_PATH.exists()},
        {"artifact": "kmeans centroids", "path": str(KMEANS_CENTROIDS), "exists": KMEANS_CENTROIDS.exists()},
        {"artifact": "mfa init centroids", "path": str(MFA_INIT_CENTROIDS), "exists": MFA_INIT_CENTROIDS.exists()},
    ]
)
display(artifacts)
assert KMEANS_ASSIGN.exists() and MFA_ASSIGN_PATH.exists(), "missing assignment artifacts"


,artifact,path,exists
0,kmeans assignments,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
1,mfa assignments,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
2,kmeans centroids,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
3,mfa init centroids,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True


### 1.1 Cluster Correspondence Check

Per-cluster deltas only make sense if cluster id `k` means the same thing in
both partitions, i.e. the nearest-centroid assignments used the exact centroids
the MFA was initialized from. Check bit-identity; if it fails, do not trust the
identity mapping below.


In [2]:
def _centroid_tensor(path: Path) -> torch.Tensor:
    obj = torch.load(path, map_location="cpu")
    if isinstance(obj, dict):
        for key in ("centroids", "mu", "means"):
            if key in obj:
                obj = obj[key]
                break
    return obj.float()


if KMEANS_CENTROIDS.exists() and MFA_INIT_CENTROIDS.exists():
    c_km = _centroid_tensor(KMEANS_CENTROIDS)
    c_init = _centroid_tensor(MFA_INIT_CENTROIDS)
    identity_ok = c_km.shape == c_init.shape and torch.equal(c_km, c_init)
    print(f"kmeans centroids: {tuple(c_km.shape)}, mfa init centroids: {tuple(c_init.shape)}")
    print(f"bit-identical: {identity_ok}")
    if not identity_ok:
        print("WARNING: centroids differ -> per-cluster deltas below are NOT meaningful under identity mapping.")
    del c_km, c_init
else:
    print("Skipped: missing centroid file(s); identity correspondence UNVERIFIED.")


kmeans centroids: (1000, 2048), mfa init centroids: (1000, 2048)
bit-identical: True


## 2. Load Assignments and Build the Contingency Matrix

Everything per-cluster derives from the K×K contingency matrix
`C[i, j] = #tokens with kmeans id i and mfa id j`, computed in one `bincount`
pass so both ~N-length assignment vectors can be freed immediately.


In [3]:
def load_assignments(path: Path) -> torch.Tensor:
    obj = torch.load(path, map_location="cpu")
    assert int(obj["K"]) == K, (obj["K"], K)
    return obj["assignments"].to(torch.long)


a_km = load_assignments(KMEANS_ASSIGN)
a_mfa = load_assignments(MFA_ASSIGN_PATH)
assert a_km.numel() == a_mfa.numel(), (a_km.numel(), a_mfa.numel())
N = a_km.numel()
print(f"Loaded {N:,} token assignments for both partitions")

contingency = (
    torch.bincount(a_km * K + a_mfa, minlength=K * K).reshape(K, K).numpy().astype(np.int64)
)
del a_km, a_mfa
gc.collect()

km_sizes = contingency.sum(axis=1)
mfa_sizes = contingency.sum(axis=0)
shared = np.diag(contingency).copy()

same_id_agreement = shared.sum() / N
print(f"same-id agreement: {same_id_agreement:.4f} "
      f"({N - shared.sum():,} of {N:,} tokens changed cluster)")


Loaded 73,687,936 token assignments for both partitions
same-id agreement: 0.5730 (31,463,838 of 73,687,936 tokens changed cluster)


### 2.1 Cluster Size Distribution

Compare the number of tokens assigned to each cluster before and after MFA
training. Both distributions use the same log-spaced bins.


In [4]:
if go is not None:
    positive_sizes = np.concatenate([km_sizes[km_sizes > 0], mfa_sizes[mfa_sizes > 0]])
    edges = np.logspace(np.log10(positive_sizes.min()), np.log10(positive_sizes.max()), 61)
    centers = (edges[:-1] + edges[1:]) / 2
    widths = np.diff(edges)

    fig = go.Figure()
    for sizes, name, color in [
        (km_sizes, "k-means (init)", COLOR_KMEANS),
        (mfa_sizes, "MFA (trained)", COLOR_MFA),
    ]:
        counts, _ = np.histogram(sizes[sizes > 0], bins=edges)
        fig.add_bar(x=centers, y=counts, width=widths, name=name, marker_color=color, opacity=0.6)

    fig.update_xaxes(type="log")
    fig.update_layout(
        title="Cluster size distribution",
        xaxis_title="assigned tokens per cluster (log-spaced bins)",
        yaxis_title="clusters",
        template=PLOT_TEMPLATE,
        barmode="overlay",
        bargap=0.05,
        width=1000,
        height=550,
        legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
    )
    save_fig(fig, "cluster_size_hist")
    fig.show()

    print(f"Zero-size clusters: k-means={(km_sizes == 0).sum()}, MFA={(mfa_sizes == 0).sum()}")


Zero-size clusters: k-means=0, MFA=0


## 3. Per-Cluster Membership Change

For each cluster id `k` (identity mapping):

- `left = km_size − shared`: tokens that were in init cluster k but are
  assigned elsewhere by the trained MFA;
- `joined = mfa_size − shared`: tokens the trained cluster k gained from other
  init clusters;
- `churn = left + joined`: total membership changes touching cluster k;
- `net_delta = mfa_size − km_size = joined − left`: net growth/shrinkage;
- normalized versions divide by the init cluster size (`rel_*`), so a
  `rel_churn` of 1 means as many tokens moved as the cluster originally held.


In [5]:
per_cluster = pd.DataFrame(
    {
        "cluster": np.arange(K),
        "km_size": km_sizes,
        "mfa_size": mfa_sizes,
        "shared": shared,
    }
)
per_cluster["left"] = per_cluster["km_size"] - per_cluster["shared"]
per_cluster["joined"] = per_cluster["mfa_size"] - per_cluster["shared"]
per_cluster["churn"] = per_cluster["left"] + per_cluster["joined"]
per_cluster["net_delta"] = per_cluster["mfa_size"] - per_cluster["km_size"]
denom = np.maximum(per_cluster["km_size"], 1)
per_cluster["rel_left"] = per_cluster["left"] / denom
per_cluster["rel_churn"] = per_cluster["churn"] / denom
per_cluster["rel_net_delta"] = per_cluster["net_delta"] / denom

display(
    per_cluster[["km_size", "mfa_size", "left", "joined", "churn", "net_delta", "rel_churn"]]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
)


,km_size,mfa_size,left,joined,churn,net_delta,rel_churn
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,73687.936000,73687.936000,31463.838000,31463.838000,62927.676000,0.000000,0.999615
std,57194.461082,54192.488363,36030.040497,28919.106295,56907.952571,32100.498257,1.071098
min,413.000000,494.000000,0.000000,1.000000,81.000000,-165902.000000,0.006210
10%,22326.100000,29209.200000,1846.800000,6488.700000,12698.700000,-37304.500000,0.226810
25%,35702.750000,43003.750000,5825.250000,13324.000000,23768.500000,-12738.500000,0.500560
50%,56763.500000,59152.500000,18209.500000,24766.500000,45049.000000,4033.500000,0.901119
75%,92937.500000,84198.750000,45996.500000,38326.750000,84923.750000,15865.500000,1.279822
90%,147550.400000,138972.300000,80884.100000,65261.600000,131780.000000,29349.500000,1.721660
max,561332.000000,539481.000000,281010.000000,273204.000000,540169.000000,159403.000000,27.430277


### 3.1 Clusters That Change the Most

Top movers by raw churn (dominated by big clusters) and by relative churn
(membership turnover regardless of size).


In [6]:
cols = ["cluster", "km_size", "mfa_size", "shared", "left", "joined", "churn", "net_delta", "rel_churn"]
display(Markdown("**Top 15 by raw churn (samples changing membership):**"))
display(per_cluster.sort_values("churn", ascending=False)[cols].head(15))

display(Markdown("**Top 15 by relative churn (churn / init size):**"))
display(per_cluster.sort_values("rel_churn", ascending=False)[cols].head(15))


**Top 15 by raw churn (samples changing membership):**

,cluster,km_size,mfa_size,shared,left,joined,churn,net_delta,rel_churn
383,383,561332,539481,280322,281010,259159,540169,-21851,0.962299
91,91,362955,522358,249154,113801,273204,387005,159403,1.066262
923,923,289206,135863,25700,263506,110163,373669,-153343,1.292051
434,434,253425,110212,24877,228548,85335,313883,-143213,1.238564
610,610,200611,198648,50379,150232,148269,298501,-1963,1.487959
940,940,143989,238383,43916,100073,194467,294540,94394,2.045573
27,27,356567,284857,175940,180627,108917,289544,-71710,0.812033
72,72,292768,233162,121344,171424,111818,283242,-59606,0.967462
500,500,166489,191156,41301,125188,149855,275043,24667,1.652019
541,541,333782,167880,113627,220155,54253,274408,-165902,0.822117


**Top 15 by relative churn (churn / init size):**

,cluster,km_size,mfa_size,shared,left,joined,churn,net_delta,rel_churn
813,813,4439,117420,48,4391,117372,121763,112981,27.430277
471,471,14351,85141,239,14112,84902,99014,70790,6.899450
703,703,9181,35318,211,8970,35107,44077,26137,4.800893
234,234,16497,63132,3576,12921,59556,72477,46635,4.393344
130,130,22487,71980,1534,20953,70446,91399,49493,4.064526
879,879,10964,51960,10396,568,41564,42132,40996,3.842758
830,830,3126,13171,2546,580,10625,11205,10045,3.584453
608,608,19218,73983,12407,6811,61576,68387,54765,3.558487
46,46,49047,140372,10243,38804,130129,168933,91325,3.444309
233,233,29926,68792,1229,28697,67563,96260,38866,3.216601


## 4. Distribution of the Membership-Change Delta

Histograms over the K clusters. `churn` is heavy-tailed, so it is shown on a
log-x axis alongside the size-normalized version; `net_delta` shows whether
training grew or shrank each cluster.


In [7]:
if px is not None:
    # px.histogram(log_x=True) bins linearly and only log-scales the axis, which
    # squashes everything into a few invisible bars; bin in log space instead.
    # churn, left and joined share bins so the three profiles are comparable.
    both = np.concatenate([per_cluster[c].to_numpy() for c in ("churn", "left", "joined")])
    both = np.clip(both, 1, None)
    edges = np.logspace(np.log10(both.min()), np.log10(both.max()), 61)
    centers = (edges[:-1] + edges[1:]) / 2
    widths = np.diff(edges)

    fig = go.Figure()
    for col, name, color in [
        ("churn", "churn (left + joined)", COLOR_NEUTRAL),
        ("left", "left (tokens lost)", COLOR_KMEANS),
        ("joined", "joined (tokens gained)", COLOR_MFA),
    ]:
        counts, _ = np.histogram(np.clip(per_cluster[col].to_numpy(), 1, None), bins=edges)
        fig.add_bar(x=centers, y=counts, width=widths, name=name, marker_color=color, opacity=0.6)
    fig.update_xaxes(type="log")
    fig.update_layout(
        title="Per-cluster membership change (churn = samples that left + samples that joined)",
        xaxis_title="tokens (log-spaced bins)",
        yaxis_title="clusters",
        template=PLOT_TEMPLATE,
        barmode="overlay",
        bargap=0.05,
        width=1000,
        height=600,
        legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
    )
    save_fig(fig, "churn_hist")
    fig.show()

    fig = px.histogram(
        per_cluster,
        x="rel_churn",
        nbins=60,
        title="Per-cluster relative membership change (churn / init cluster size)",
        color_discrete_sequence=[COLOR_NEUTRAL],
        template=PLOT_TEMPLATE,
    )
    fig.add_vline(x=1.0, line_dash="dash", line_color="#0b0b0b",
                  annotation_text="churn = init size", annotation_position="top right")
    fig.update_layout(xaxis_title="relative churn", yaxis_title="clusters", bargap=0.05)
    save_fig(fig, "rel_churn_hist")
    fig.show()



### 4.1 Init → Trained Flow Heatmap

Cell `(i, j)` counts the tokens that **left** k-means cluster `i` and
**joined** MFA cluster `j` — i.e. the off-diagonal contingency entry
`C[i, j]`. The diagonal (tokens that stayed in their cluster) is zeroed so it
does not swamp the color scale, and counts are shown as `log10(tokens + 1)`.
Hot rows are init clusters that training drained broadly; hot columns are
trained clusters that absorbed tokens from many init clusters.


In [8]:
# if go is not None:
#     flow = contingency.astype(np.float32).copy()
#     np.fill_diagonal(flow, 0.0)  # diagonal = retained tokens, not membership change
#     fig = go.Figure(
#         go.Heatmap(
#             z=np.log10(flow + 1.0),
#             colorscale="Viridis",
#             zmin=0,
#             zmax=4,
#             colorbar=dict(title="log10(tokens + 1)"),
#             hovertemplate="kmeans i=%{y}<br>mfa j=%{x}<br>log10(tokens+1)=%{z:.2f}<extra></extra>",
#         )
#     )
#     fig.update_layout(
#         title="Membership flow: tokens leaving k-means cluster i and joining MFA cluster j",
#         xaxis_title="MFA cluster j (trained)",
#         yaxis_title="k-means cluster i (init)",
#         yaxis=dict(autorange="reversed"),
#         template=PLOT_TEMPLATE,
#         width=820,
#         height=780,
#     )
#     save_fig(fig, "flow_heatmap")
#     fig.show()
#     del flow


### 4.3 Clustered Flow Heatmap (clustermap)

Same matrix as 4.2, but rows and columns are reordered by hierarchical
clustering (average linkage, Euclidean distance on the `log10(tokens+1)`
flow profiles) so that init clusters draining to similar destinations sit
next to each other, and trained clusters absorbing from similar sources sit
next to each other. Axis positions no longer correspond to cluster ids —
hover to see the true `(i, j)` pair. Blocks of bright cells are groups of
init clusters whose tokens training redistributed into a common set of
trained clusters.


In [9]:
# if go is not None:
#     from scipy.cluster.hierarchy import leaves_list, linkage

#     flow = contingency.astype(np.float32).copy()
#     np.fill_diagonal(flow, 0.0)
#     log_flow = np.log10(flow + 1.0)

#     # euclidean rather than cosine: clusters with zero outflow/inflow produce
#     # all-zero profiles, for which cosine distance is undefined
#     row_order = leaves_list(linkage(log_flow, method="average", metric="euclidean"))
#     col_order = leaves_list(linkage(log_flow.T, method="average", metric="euclidean"))
#     z = log_flow[np.ix_(row_order, col_order)]

#     fig = go.Figure(
#         go.Heatmap(
#             z=z,
#             x=[str(j) for j in col_order],
#             y=[str(i) for i in row_order],
#             colorscale="Viridis",
#             zmin=0,
#             zmax=4,
#             colorbar=dict(title="log10(tokens + 1)"),
#             hovertemplate="kmeans i=%{y}<br>mfa j=%{x}<br>log10(tokens+1)=%{z:.2f}<extra></extra>",
#         )
#     )
#     fig.update_layout(
#         title="Clustered membership flow (rows/cols reordered by hierarchical clustering)",
#         xaxis=dict(title="MFA cluster j (trained, clustered order)", showticklabels=False),
#         yaxis=dict(title="k-means cluster i (init, clustered order)", showticklabels=False, autorange="reversed"),
#         template=PLOT_TEMPLATE,
#         width=820,
#         height=780,
#     )
#     save_fig(fig, "flow_clustermap")
#     fig.show()
#     del flow, log_flow, z


## 4.4 Tokens Exchanged vs Centroid Distance

Does a cluster exchange more tokens with clusters that are near it, or far from
it? Each point below is a cluster; we correlate **how many tokens it exchanges**
against the **token-weighted mean centroid distance to its exchange partners**.
Each partner is weighted by the number of tokens actually exchanged with it
(`Σ_p C[k,p]·d(k,p) / Σ_p C[k,p]`), so the y-axis reflects where the token
*mass* goes — not a rare 1-token leak to a distant cluster — and is on the same
token-weighted footing as the x-axis. Stratified into:

- **give**: tokens `k` gave away (`left`) vs distance to partners `j`, weighted by `C[k,j]`;
- **take**: tokens `k` took in (`joined`) vs distance to partners `i`, weighted by `C[i,k]`;
- **give+take**: total `churn` vs distance to all partners, weighted by `C[k,p]+C[p,k]`.

The first cell computes the weighted mean partner distances (under identity
cluster correspondence, with both the **k-means centroids** and the **MFA
means** as cluster centers); the scatter cell plots tokens exchanged (x) against
that distance (y) with a Spearman ρ per stratum.


In [10]:
from scipy.spatial.distance import cdist

from dalg.models.mfa import load_mfa

# cluster centers under identity correspondence
c_km = _centroid_tensor(KMEANS_CENTROIDS).numpy()
_model = load_mfa(MFA_RUN / "mfa_model.pt", map_location="cpu")
mu = _model.mu.detach().float().cpu().numpy()
del _model
gc.collect()
assert c_km.shape == mu.shape == (K, c_km.shape[1])

# off-diagonal flow -> per-partner token weights (match the x-axis token counts)
_flow = contingency.astype(np.float64).copy()
np.fill_diagonal(_flow, 0.0)
W_give = _flow                 # W_give[k, j] = tokens k -> j        (rowsum = left)
W_take = _flow.T               # W_take[k, i] = tokens i -> k        (rowsum = joined)
W_both = W_give + W_take       # total tokens exchanged with partner (rowsum = churn)


def weighted_partner_mean(values: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """Token-weighted mean of values[k, p] over partners p (weight = tokens exchanged)."""
    denom = weights.sum(axis=1)
    num = (values * weights).sum(axis=1)
    return np.where(denom > 0, num / np.maximum(denom, 1), np.nan)


exchange = pd.DataFrame({"cluster": np.arange(K)})
exchange["n_give"] = (W_give > 0).sum(axis=1)
exchange["n_take"] = (W_take > 0).sum(axis=1)
exchange["n_both"] = (W_both > 0).sum(axis=1)
for label, centers in [("km", c_km), ("mfa", mu)]:
    D = cdist(centers, centers)
    exchange[f"give_dist_{label}"] = weighted_partner_mean(D, W_give)
    exchange[f"take_dist_{label}"] = weighted_partner_mean(D, W_take)
    exchange[f"both_dist_{label}"] = weighted_partner_mean(D, W_both)

per_cluster = per_cluster.merge(exchange, on="cluster")

display(
    exchange[
        ["n_give", "n_take", "n_both",
         "give_dist_km", "take_dist_km", "both_dist_km",
         "give_dist_mfa", "take_dist_mfa", "both_dist_mfa"]
    ].describe(percentiles=[0.1, 0.5, 0.9])
)


,n_give,n_take,n_both,give_dist_km,take_dist_km,both_dist_km,give_dist_mfa,take_dist_mfa,both_dist_mfa
count,1000.000000,1000.000000,1000.000000,996.000000,1000.000000,1000.000000,996.000000,1000.000000,1000.000000
mean,211.957000,211.957000,322.650000,33.097719,32.529971,32.853211,35.711757,35.849757,35.732528
std,166.717161,133.255725,177.433007,9.852537,10.593080,10.449143,9.080008,10.148763,9.952604
min,0.000000,1.000000,1.000000,11.896355,11.495596,11.956036,10.266587,10.943370,10.762726
10%,43.000000,43.000000,96.000000,24.556227,22.312218,23.858032,26.286584,25.609541,26.144317
50%,154.000000,222.000000,316.500000,31.157231,30.725237,30.784236,34.840190,34.901202,34.837229
90%,488.200000,361.000000,573.100000,44.822578,45.061477,44.565554,45.515295,47.157370,45.460446
max,703.000000,846.000000,870.000000,134.533300,98.434606,128.172830,119.965035,109.683188,116.465965


In [11]:
if px is not None:
    from scipy.stats import spearmanr

    # per-cluster: x = tokens exchanged, y = weighted mean centroid distance to partners
    strata = [
        ("give", "left", COLOR_KMEANS),        # tokens k gave away
        ("take", "joined", COLOR_MFA),         # tokens k took in
        ("give+take", "churn", COLOR_NEUTRAL), # both
    ]
    for label, title in [("km", "k-means centroid geometry"), ("mfa", "MFA mean geometry")]:
        fig = go.Figure()
        for name, xcol, color in strata:
            ycol = f"{'both' if name == 'give+take' else name}_dist_{label}"
            sub = per_cluster[["cluster", xcol, ycol]].dropna()
            sub = sub[sub[xcol] > 0]
            x, y, cl = sub[xcol].to_numpy(), sub[ycol].to_numpy(), sub["cluster"].to_numpy()
            rho, _ = spearmanr(x, y)
            fig.add_scatter(
                x=x, y=y, mode="markers",
                name=f"{name} (ρ={rho:.2f})",
                marker=dict(color=color, size=5, opacity=0.5),
                customdata=cl,
                hovertemplate="cluster %{customdata}<br>exchanged=%{x:,}<br>distance=%{y:.2f}<extra></extra>",
            )
            # least-squares fit y ~ a*log10(x) + b (linear in the plotted log-x axis)
            lx = np.log10(x)
            a, b = np.polyfit(lx, y, 1)
            xs = np.linspace(lx.min(), lx.max(), 100)
            fig.add_scatter(
                x=10 ** xs, y=a * xs + b, mode="lines",
                line=dict(color=color, width=2),
                name=f"{name} fit", showlegend=False, hoverinfo="skip",
            )
        fig.update_xaxes(type="log")
        fig.update_layout(
            title=f"Tokens exchanged vs mean centroid distance to partners ({title})",
            xaxis_title="tokens exchanged (log)",
            yaxis_title="mean centroid distance to exchange partners",
            template=PLOT_TEMPLATE,
            width=1000,
            height=550,
            legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
        )
        save_fig(fig, f"exchange_distance_vs_count_{label}")
        fig.show()


## 4.5 Tokens Exchanged vs Intrinsic-Dimension Gap

Same construction as 4.4, but the y-axis replaces centroid distance with the
token-weighted mean **intrinsic-dimension gap** to the exchange partners: for
cluster `k`, `Σ_p C[k,p]·(ID[p] − ID[k]) / Σ_p C[k,p]` over its partners `p`
(each weighted by tokens exchanged). A positive value means `k`'s exchanged
tokens go mostly to/from clusters of *higher* intrinsic dimension than itself.

Stratified the same way — **give** vs `left`, **take** vs `joined`,
**give+take** vs `churn` (same token weights as 4.4) — and computed for both
intrinsic-dimension sources: the **k-means partition** IDs
(`centroids_1000_05/intrinsic_dims.pt`) and the **MFA partition** IDs
(`1000_05_10/intrinsic_dims.pt`), under identity cluster correspondence.


In [12]:
# intrinsic-dim per cluster from each partition (identity cluster correspondence)
ID_PATHS = {
    "km": CENTROIDS_DIR / "intrinsic_dims.pt",
    "mfa": MFA_RUN / "intrinsic_dims.pt",
}
ids_by_source = {}
for label, path in ID_PATHS.items():
    obj = torch.load(path, map_location="cpu")
    assert int(obj["K"]) == K, (label, obj["K"], K)
    ids_by_source[label] = obj["intrinsic_dims"].to(torch.float64).numpy()


def weighted_partner_id_gap(ids: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """Token-weighted mean over partners of (partner_ID - k_ID); ID==0 = invalid partner."""
    valid = ids > 0
    diff = ids[None, :] - ids[:, None]     # diff[k, p] = ID[p] - ID[k]
    w = weights * valid[None, :]           # drop invalid partners from the weighting
    denom = w.sum(axis=1)
    num = (diff * w).sum(axis=1)
    return np.where((denom > 0) & valid, num / np.maximum(denom, 1), np.nan)


# W_give / W_take / W_both come from the 4.4 exchange cell (token weights)
for label, ids in ids_by_source.items():
    per_cluster[f"give_iddiff_{label}"] = weighted_partner_id_gap(ids, W_give)
    per_cluster[f"take_iddiff_{label}"] = weighted_partner_id_gap(ids, W_take)
    per_cluster[f"both_iddiff_{label}"] = weighted_partner_id_gap(ids, W_both)

display(
    per_cluster[
        ["give_iddiff_km", "take_iddiff_km", "both_iddiff_km",
         "give_iddiff_mfa", "take_iddiff_mfa", "both_iddiff_mfa"]
    ].describe(percentiles=[0.1, 0.5, 0.9])
)


,give_iddiff_km,take_iddiff_km,both_iddiff_km,give_iddiff_mfa,take_iddiff_mfa,both_iddiff_mfa
count,996.000000,1000.000000,1000.000000,996.000000,1000.000000,1000.000000
mean,16.008076,56.522885,41.264164,18.410342,16.664725,20.178947
std,103.360662,111.728029,111.580751,85.694793,82.537515,83.140181
min,-334.850843,-383.158897,-302.298890,-404.704657,-403.396537,-404.030476
10%,-110.875204,-80.030117,-91.131021,-75.494981,-76.811886,-73.733542
50%,13.928413,52.754488,36.166581,13.921707,17.116153,16.607359
90%,141.376470,191.239093,175.044464,122.231429,110.503460,119.420730
max,384.542670,567.710256,555.855037,429.977704,387.983548,391.773547


In [13]:
if px is not None:
    from scipy.stats import spearmanr

    strata = [
        ("give", "left", COLOR_KMEANS),
        ("take", "joined", COLOR_MFA),
        ("give+take", "churn", COLOR_NEUTRAL),
    ]
    for label, title in [
        ("km", "k-means partition intrinsic dims"),
        ("mfa", "MFA partition intrinsic dims"),
    ]:
        fig = go.Figure()
        for name, xcol, color in strata:
            ycol = f"{'both' if name == 'give+take' else name}_iddiff_{label}"
            sub = per_cluster[["cluster", xcol, ycol]].dropna()
            sub = sub[sub[xcol] > 0]
            x, y, cl = sub[xcol].to_numpy(), sub[ycol].to_numpy(), sub["cluster"].to_numpy()
            rho, _ = spearmanr(x, y)
            fig.add_scatter(
                x=x, y=y, mode="markers",
                name=f"{name} (ρ={rho:.2f})",
                marker=dict(color=color, size=5, opacity=0.5),
                customdata=cl,
                hovertemplate="cluster %{customdata}<br>exchanged=%{x:,}<br>ID gap=%{y:.1f}<extra></extra>",
            )
            # least-squares fit y ~ a*log10(x) + b (linear in the plotted log-x axis)
            lx = np.log10(x)
            a, b = np.polyfit(lx, y, 1)
            xs = np.linspace(lx.min(), lx.max(), 100)
            fig.add_scatter(
                x=10 ** xs, y=a * xs + b, mode="lines",
                line=dict(color=color, width=2),
                name=f"{name} fit", showlegend=False, hoverinfo="skip",
            )
        fig.add_hline(y=0, line_dash="dash", line_color="#0b0b0b")
        fig.update_xaxes(type="log")
        fig.update_layout(
            title=f"Tokens exchanged vs mean intrinsic-dim gap to partners ({title})",
            xaxis_title="tokens exchanged (log)",
            yaxis_title="mean (partner ID − cluster ID) over exchange partners",
            template=PLOT_TEMPLATE,
            width=1000,
            height=550,
            legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
        )
        save_fig(fig, f"exchange_iddiff_vs_count_{label}")
        fig.show()


## 4.6 Outlier Samples in the K-Means Exchange Plots

Each plotted sample is a `(cluster, stratum)` pair, where the stratum is
**give**, **take**, or **give+take**. Within each stratum, outlierness is the
absolute residual from the same linear fit used in the plots,
`y ~ log10(tokens exchanged)`, robustly standardized by the residual median
and median absolute deviation (MAD). The tables show the top
`OUTLIERS_PER_STRATUM` samples from each stratum for the two K-means plots,
followed by samples that are outliers in both plots. A signed residual says
whether the observed value lies above or below its fitted value.


In [14]:
OUTLIERS_PER_STRATUM = 10
OUTLIER_STRATA = [
    ("give", "left", "give_dist_km", "give_iddiff_km"),
    ("take", "joined", "take_dist_km", "take_iddiff_km"),
    ("give+take", "churn", "both_dist_km", "both_iddiff_km"),
]


def ranked_plot_outliers(
    frame: pd.DataFrame,
    *,
    stratum: str,
    x_col: str,
    y_col: str,
    metric_name: str,
) -> pd.DataFrame:
    """Rank samples by robustly standardized residual from y ~ log10(x)."""
    sub = frame[["cluster", x_col, y_col]].dropna().copy()
    sub = sub[sub[x_col] > 0]

    log_x = np.log10(sub[x_col].to_numpy(dtype=np.float64))
    y = sub[y_col].to_numpy(dtype=np.float64)
    slope, intercept = np.polyfit(log_x, y, 1)
    expected = slope * log_x + intercept
    residual = y - expected

    residual_center = np.median(residual)
    mad = np.median(np.abs(residual - residual_center))
    robust_scale = 1.4826 * mad
    if not np.isfinite(robust_scale) or robust_scale <= np.finfo(float).eps:
        robust_scale = np.std(residual, ddof=1)
    if not np.isfinite(robust_scale) or robust_scale <= np.finfo(float).eps:
        robust_scale = 1.0

    sub["stratum"] = stratum
    sub["tokens_exchanged"] = sub[x_col].astype(np.int64)
    sub[f"{metric_name}_expected"] = expected
    sub[f"{metric_name}_residual"] = residual
    sub[f"{metric_name}_outlier_score"] = np.abs(
        (residual - residual_center) / robust_scale
    )
    sub = sub.nlargest(OUTLIERS_PER_STRATUM, f"{metric_name}_outlier_score").copy()
    sub[f"{metric_name}_rank"] = np.arange(1, len(sub) + 1)
    return sub.rename(columns={y_col: metric_name})[
        ["cluster", "stratum", "tokens_exchanged", metric_name,
         f"{metric_name}_expected", f"{metric_name}_residual",
         f"{metric_name}_outlier_score", f"{metric_name}_rank"]
    ]


distance_outliers = pd.concat(
    [
        ranked_plot_outliers(
            per_cluster, stratum=stratum, x_col=x_col, y_col=distance_col,
            metric_name="mean_centroid_distance_km",
        )
        for stratum, x_col, distance_col, _ in OUTLIER_STRATA
    ],
    ignore_index=True,
).sort_values("mean_centroid_distance_km_outlier_score", ascending=False)

id_gap_outliers = pd.concat(
    [
        ranked_plot_outliers(
            per_cluster, stratum=stratum, x_col=x_col, y_col=id_gap_col,
            metric_name="mean_intrinsic_dim_gap_km",
        )
        for stratum, x_col, _, id_gap_col in OUTLIER_STRATA
    ],
    ignore_index=True,
).sort_values("mean_intrinsic_dim_gap_km_outlier_score", ascending=False)

overlap_outliers = distance_outliers.merge(
    id_gap_outliers,
    on=["cluster", "stratum", "tokens_exchanged"],
    how="inner",
)
overlap_outliers["joint_outlier_score"] = (
    overlap_outliers["mean_centroid_distance_km_outlier_score"]
    + overlap_outliers["mean_intrinsic_dim_gap_km_outlier_score"]
)
overlap_outliers = overlap_outliers.sort_values("joint_outlier_score", ascending=False)

display(Markdown("**Outliers: tokens exchanged vs mean centroid distance (K-means geometry)**"))
display(distance_outliers)
display(Markdown("**Outliers: tokens exchanged vs mean intrinsic-dimension gap (K-means partition)**"))
display(id_gap_outliers)
display(Markdown("**Outliers appearing in both K-means plots**"))
display(overlap_outliers)


**Outliers: tokens exchanged vs mean centroid distance (K-means geometry)**

,cluster,stratum,tokens_exchanged,mean_centroid_distance_km,mean_centroid_distance_km_expected,mean_centroid_distance_km_residual,mean_centroid_distance_km_outlier_score,mean_centroid_distance_km_rank
0,401,give,2733,134.533300,39.580275,94.953024,16.710179,1
20,401,give+take,3282,128.172830,50.863356,77.309474,13.594376,1
1,485,give,13862,91.178719,33.270493,57.908226,10.256540,2
10,53,take,3062,98.434606,43.685025,54.749581,9.125836,1
2,53,give,158,101.622460,50.657348,50.965112,9.046968,3
21,485,give+take,14477,90.257406,40.400602,49.856804,8.836292,2
22,53,give+take,3220,98.591029,50.997808,47.593221,8.443969,3
11,830,take,10625,80.857728,36.464635,44.393093,7.444156,2
12,401,take,549,96.509508,53.659669,42.849839,7.193564,3
3,576,give,4746,76.696203,37.435622,39.260581,7.007901,4


**Outliers: tokens exchanged vs mean intrinsic-dimension gap (K-means partition)**

,cluster,stratum,tokens_exchanged,mean_intrinsic_dim_gap_km,mean_intrinsic_dim_gap_km_expected,mean_intrinsic_dim_gap_km_residual,mean_intrinsic_dim_gap_km_outlier_score,mean_intrinsic_dim_gap_km_rank
10,491,take,3625,-383.158897,98.352607,-481.511504,4.930252,1
11,454,take,6506,559.823701,84.399216,475.424486,4.903135,2
12,87,take,7762,509.162458,80.188052,428.974407,4.425818,3
13,84,take,390,567.710256,151.541418,416.168838,4.294229,4
14,897,take,8998,-332.954323,76.662852,-409.617175,4.191473,5
20,471,give+take,99014,368.002303,-5.791315,373.793618,4.188360,1
21,454,give+take,7684,507.213561,135.360823,371.852738,4.166468,2
22,87,give+take,8219,502.233240,131.643976,370.589264,4.152216,3
15,471,take,84902,424.139820,23.115408,401.024412,4.138606,6
23,491,give+take,9148,-233.274596,125.730515,-359.005111,4.077211,4


**Outliers appearing in both K-means plots**

,cluster,stratum,tokens_exchanged,mean_centroid_distance_km,mean_centroid_distance_km_expected,mean_centroid_distance_km_residual,mean_centroid_distance_km_outlier_score,mean_centroid_distance_km_rank,mean_intrinsic_dim_gap_km,mean_intrinsic_dim_gap_km_expected,mean_intrinsic_dim_gap_km_residual,mean_intrinsic_dim_gap_km_outlier_score,mean_intrinsic_dim_gap_km_rank,joint_outlier_score
0,401,give,2733,134.533300,39.580275,94.953024,16.710179,1,349.222466,75.226657,273.995809,3.432310,8,20.142489
1,830,take,10625,80.857728,36.464635,44.393093,7.444156,2,451.818541,72.697604,379.120937,3.913528,8,11.357685
3,84,take,390,97.022051,55.644190,41.377861,6.954545,4,567.710256,151.541418,416.168838,4.294229,4,11.248774
2,576,give,4746,76.696203,37.435622,39.260581,7.007901,4,380.475558,55.635113,324.840446,4.074373,1,11.082274
4,830,give+take,11205,80.901483,42.206784,38.694700,6.901682,4,448.003570,114.530279,333.473291,3.733568,6,10.635250
6,471,take,84902,59.338921,24.403264,34.935657,5.908464,9,424.139820,23.115408,401.024412,4.138606,6,10.047071
5,781,give,18185,68.347662,32.215674,36.131988,6.462863,5,-261.402914,7.950542,-269.353457,3.429072,9,9.891935
7,471,give+take,99014,57.859299,26.845838,31.013461,5.570373,8,368.002303,-5.791315,373.793618,4.188360,1,9.758733


## 5. High-Churn vs Low-Churn Spectral Change

Use the PCA eigenvalue spectra already saved in the K-means and MFA
`intrinsic_dims.pt` artifacts. Clusters are eligible only when both artifacts
contain a nonempty spectrum estimated from exactly the same number of sampled
activations. No assignments or activation shards are loaded here.

High-churn clusters are the top 10% by
`relative churn = (left + joined) / K-means size`. Controls are selected without
replacement from the bottom half by churn and matched by K-means cluster size.
From each normalized saved spectrum `p`, report:

- intrinsic dimension at the saved variance threshold (normally 90%);
- effective rank, `exp(-sum(p * log(p)))`;
- variance explained by the first 10 PCs;
- spectral isotropy, defined here as normalized participation ratio
  `1 / (len(p) * sum(p**2))` (1 is a flat spectrum);
- normalized area between the K-means and MFA cumulative-variance curves.

Changes are `MFA - K-means`. Thus negative dimension/effective-rank/isotropy
together with positive first-10-PC variance means the trained membership is more
spectrally concentrated; the opposite pattern means it is more diffuse. Confidence
intervals below bootstrap the size-matched cluster pairs, treating each saved PCA
spectrum as one cluster-level estimate.


In [15]:
from scipy.optimize import linear_sum_assignment

HIGH_CHURN_FRACTION = 0.10
LOW_CHURN_POOL_FRACTION = 0.50

SPECTRAL_ID_PATHS = {
    "km": CENTROIDS_DIR / "intrinsic_dims.pt",
    "mfa": MFA_RUN / "intrinsic_dims.pt",
}
id_data = {
    label: torch.load(path, map_location="cpu", weights_only=True)
    for label, path in SPECTRAL_ID_PATHS.items()
}
km_id, mfa_id = id_data["km"], id_data["mfa"]
assert int(km_id["K"]) == int(mfa_id["K"]) == K
assert np.isclose(km_id["variance_threshold"], mfa_id["variance_threshold"])
ID_THRESHOLD = float(km_id["variance_threshold"])

km_sample_sizes = km_id["sample_sizes"].numpy()
mfa_sample_sizes = mfa_id["sample_sizes"].numpy()
valid = np.array([
    km_sample_sizes[k] == mfa_sample_sizes[k]
    and km_sample_sizes[k] >= 2
    and km_id["cluster_variances"][k].numel() > 0
    and km_id["cluster_variances"][k].numel() == mfa_id["cluster_variances"][k].numel()
    for k in range(K)
])

def normalized_spectrum(values: torch.Tensor) -> np.ndarray:
    p = values.double().clamp_min(0).numpy()
    total = p.sum()
    if not np.isfinite(total) or total <= 0:
        raise ValueError("PCA spectrum has non-positive or non-finite total variance")
    return p / total

def spectrum_metrics(p: np.ndarray) -> dict[str, float]:
    cumulative = np.cumsum(p)
    positive = p > 0
    entropy = -(p[positive] * np.log(p[positive])).sum()
    return {
        "intrinsic_dim": float(np.searchsorted(cumulative, ID_THRESHOLD) + 1),
        "effective_rank": float(np.exp(entropy)),
        "var_first10": float(p[:10].sum()),
        "isotropy": float(1.0 / (len(p) * np.square(p).sum())),
    }

rows = []
for cluster in np.flatnonzero(valid):
    p_km = normalized_spectrum(km_id["cluster_variances"][cluster])
    p_mfa = normalized_spectrum(mfa_id["cluster_variances"][cluster])
    km_metrics = spectrum_metrics(p_km)
    mfa_metrics = spectrum_metrics(p_mfa)
    assert int(km_metrics["intrinsic_dim"]) == int(km_id["intrinsic_dims"][cluster])
    assert int(mfa_metrics["intrinsic_dim"]) == int(mfa_id["intrinsic_dims"][cluster])
    curve_gap = np.r_[0.0, np.abs(np.cumsum(p_km) - np.cumsum(p_mfa))]
    row = {
        "cluster": int(cluster),
        "pca_sample_n": int(km_sample_sizes[cluster]),
        "cumulative_variance_area": float(
            np.trapezoid(curve_gap, dx=1.0 / (len(curve_gap) - 1))
        ),
    }
    for metric in km_metrics:
        row[f"km_{metric}"] = km_metrics[metric]
        row[f"mfa_{metric}"] = mfa_metrics[metric]
        row[f"delta_{metric}"] = mfa_metrics[metric] - km_metrics[metric]
    rows.append(row)

spectral_by_cluster = per_cluster.merge(pd.DataFrame(rows), on="cluster", how="inner")
SPECTRAL_CHANGE_METRICS = [
    "delta_intrinsic_dim",
    "delta_effective_rank",
    "delta_var_first10",
    "delta_isotropy",
    "cumulative_variance_area",
]

n_group = max(1, int(np.ceil(HIGH_CHURN_FRACTION * len(spectral_by_cluster))))
ranked = spectral_by_cluster.sort_values(["rel_churn", "cluster"])
high = ranked.tail(n_group).copy()
low_pool = ranked.head(int(np.ceil(LOW_CHURN_POOL_FRACTION * len(ranked)))).copy()
assert set(high["cluster"]).isdisjoint(low_pool["cluster"])

cost = np.abs(
    np.log1p(high["km_size"].to_numpy())[:, None]
    - np.log1p(low_pool["km_size"].to_numpy())[None, :]
)
# Enforce the same saved PCA sample count across every matched high/low pair.
cost += 1e6 * (
    high["pca_sample_n"].to_numpy()[:, None]
    != low_pool["pca_sample_n"].to_numpy()[None, :]
)
high_row, low_row = linear_sum_assignment(cost)
high_selected = high.iloc[high_row].copy().reset_index(drop=True)
low_selected = low_pool.iloc[low_row].copy().reset_index(drop=True)
assert np.array_equal(high_selected["pca_sample_n"], low_selected["pca_sample_n"])

matches = pd.DataFrame({
    "match_id": np.arange(n_group),
    "high_cluster": high_selected["cluster"],
    "low_cluster": low_selected["cluster"],
    "high_rel_churn": high_selected["rel_churn"],
    "low_rel_churn": low_selected["rel_churn"],
    "high_km_size": high_selected["km_size"],
    "low_km_size": low_selected["km_size"],
    "pca_sample_n": high_selected["pca_sample_n"],
})
matches["log_size_gap"] = np.abs(
    np.log1p(matches["high_km_size"]) - np.log1p(matches["low_km_size"])
)
high_selected["match_id"], high_selected["group"] = matches["match_id"], "high"
low_selected["match_id"], low_selected["group"] = matches["match_id"], "low"
matched_spectral = pd.concat([high_selected, low_selected], ignore_index=True)

print(
    f"Equal-sample valid spectra: {len(spectral_by_cluster)}/{K}; "
    f"high/low matched clusters: {n_group} each"
)
display(matches.describe(percentiles=[0.1, 0.5, 0.9]))


Equal-sample valid spectra: 979/1000; high/low matched clusters: 98 each


,match_id,high_cluster,low_cluster,high_rel_churn,low_rel_churn,high_km_size,low_km_size,pca_sample_n,log_size_gap
count,98.000000,98.000000,98.000000,98.000000,98.000000,98.000000,98.000000,98.0,98.000000
mean,48.500000,547.000000,503.571429,2.336909,0.543831,37842.469388,40681.979592,10000.0,0.127472
std,28.434134,287.143773,283.403032,0.739112,0.238795,25950.906233,24137.598812,0.0,0.245268
min,0.000000,9.000000,37.000000,1.717116,0.036805,10064.000000,12779.000000,10000.0,0.000026
10%,9.700000,101.700000,114.500000,1.773740,0.196591,14943.700000,19155.400000,10000.0,0.000568
50%,48.500000,564.500000,515.500000,2.070160,0.566140,29290.000000,32190.000000,10000.0,0.007181
90%,87.300000,921.900000,877.900000,3.128281,0.862528,68466.300000,68354.400000,10000.0,0.526922
max,97.000000,990.000000,992.000000,6.899450,0.892916,143989.000000,142872.000000,10000.0,1.090279


### 5.1 Are Low Top-q K-Means Spectra Also High-Churn?

Reproduce the left-tail selection from `kmeans_mfa_comparison.ipynb`: the
bottom 10% of K-means clusters by variance fraction captured in the first
`q=Q` PCs. Compare those cluster IDs with the high-churn group already defined
above (top 10% by relative churn). The overlap is reported against its random
expectation, with a one-sided Fisher exact test for enrichment.


In [16]:
from scipy.stats import fisher_exact

KMEANS_LEFT_TAIL_FRACTION = 0.10
comparison_valid = (km_id["intrinsic_dims"] > 0) & (mfa_id["intrinsic_dims"] > 0)
kmeans_topq_rows = []
for cluster in torch.nonzero(comparison_valid, as_tuple=True)[0].tolist():
    p_km = normalized_spectrum(km_id["cluster_variances"][cluster])
    kmeans_topq_rows.append(
        {"cluster": cluster, "km_var_first10": float(p_km[:Q].sum())}
    )
kmeans_topq_by_cluster = pd.DataFrame(kmeans_topq_rows)
kmeans_left_tail_threshold = float(
    kmeans_topq_by_cluster["km_var_first10"].quantile(KMEANS_LEFT_TAIL_FRACTION)
)
kmeans_left_tail = kmeans_topq_by_cluster.loc[
    kmeans_topq_by_cluster["km_var_first10"] <= kmeans_left_tail_threshold
].merge(per_cluster[["cluster", "rel_churn"]], on="cluster", how="left")
kmeans_left_tail = kmeans_left_tail.sort_values(["km_var_first10", "cluster"])
kmeans_left_tail_clusters = kmeans_left_tail["cluster"].astype(int).tolist()
high_churn_clusters = sorted(high["cluster"].astype(int).tolist())
left_tail_high_churn_clusters = sorted(
    set(kmeans_left_tail_clusters).intersection(high_churn_clusters)
)

eligible_n = len(kmeans_topq_by_cluster)
assert set(high_churn_clusters).issubset(kmeans_topq_by_cluster["cluster"])
overlap_n = len(left_tail_high_churn_clusters)
left_only_n = len(kmeans_left_tail_clusters) - overlap_n
high_only_n = len(high_churn_clusters) - overlap_n
neither_n = eligible_n - overlap_n - left_only_n - high_only_n
tail_churn_contingency = pd.DataFrame(
    [[overlap_n, left_only_n], [high_only_n, neither_n]],
    index=["left spectral tail", "not left spectral tail"],
    columns=["high churn", "not high churn"],
)
odds_ratio, fisher_p = fisher_exact(
    tail_churn_contingency.to_numpy(), alternative="greater"
)
expected_overlap = (
    len(kmeans_left_tail_clusters) * len(high_churn_clusters) / eligible_n
)
enrichment = overlap_n / expected_overlap if expected_overlap > 0 else np.nan

tail_churn_summary = pd.DataFrame(
    [
        {
            "eligible_clusters": eligible_n,
            "left_tail_clusters": len(kmeans_left_tail_clusters),
            "high_churn_clusters": len(high_churn_clusters),
            "overlap": overlap_n,
            "expected_overlap": expected_overlap,
            "enrichment_vs_random": enrichment,
            "left_tail_high_churn_rate": overlap_n / len(kmeans_left_tail_clusters),
            "non_tail_high_churn_rate": high_only_n / (eligible_n - len(kmeans_left_tail_clusters)),
            "fisher_odds_ratio": odds_ratio,
            "fisher_p_greater": fisher_p,
        }
    ]
)
kmeans_left_tail["high_churn"] = kmeans_left_tail["cluster"].isin(
    left_tail_high_churn_clusters
)

print("K-means left-tail clusters that are also high-churn:")
print(left_tail_high_churn_clusters)
display(tail_churn_summary)
display(tail_churn_contingency)
display(
    kmeans_left_tail[
        ["cluster", "km_var_first10", "rel_churn", "high_churn"]
    ].reset_index(drop=True)
)


K-means left-tail clusters that are also high-churn:
[12, 25, 36, 52, 56, 66, 117, 130, 233, 315, 321, 358, 360, 478, 479, 497, 503, 508, 510, 516, 570, 578, 599, 620, 688, 752, 772, 781, 793, 899, 976]


,eligible_clusters,left_tail_clusters,high_churn_clusters,overlap,expected_overlap,enrichment_vs_random,left_tail_high_churn_rate,non_tail_high_churn_rate,fisher_odds_ratio,fisher_p_greater
0,1000,100,98,31,9.8,3.163265,0.31,0.074444,5.585767,1.951711e-10


,high churn,not high churn
left spectral tail,31,69
not left spectral tail,67,833


,cluster,km_var_first10,rel_churn,high_churn
0,479,0.109036,2.107667,True
1,66,0.112455,1.912082,True
2,12,0.113946,2.670730,True
3,693,0.114405,1.403981,False
4,897,0.114468,1.149252,False
...,...,...,...,...
95,378,0.176987,1.482915,False
96,508,0.177752,1.783546,True
97,293,0.178099,1.607910,False
98,758,0.180478,1.256238,False


In [17]:
BOOTSTRAP_REPS = 10_000
SPECTRAL_SEED = 20260715
rng = np.random.default_rng(SPECTRAL_SEED)

high_values = high_selected[SPECTRAL_CHANGE_METRICS].to_numpy(dtype=np.float64)
low_values = low_selected[SPECTRAL_CHANGE_METRICS].to_numpy(dtype=np.float64)
n_pairs = len(matches)
bootstrap_indices = rng.integers(0, n_pairs, size=(BOOTSTRAP_REPS, n_pairs))

group_comparison_rows = []
for metric_i, metric in enumerate(SPECTRAL_CHANGE_METRICS):
    high_boot = high_values[bootstrap_indices, metric_i].mean(axis=1)
    low_boot = low_values[bootstrap_indices, metric_i].mean(axis=1)
    diff_boot = high_boot - low_boot
    p_two_sided = min(
        1.0, 2 * min(np.mean(diff_boot <= 0), np.mean(diff_boot >= 0))
    )
    group_comparison_rows.append({
        "metric": metric,
        "high_mean": high_values[:, metric_i].mean(),
        "low_mean": low_values[:, metric_i].mean(),
        "high_minus_low": high_values[:, metric_i].mean() - low_values[:, metric_i].mean(),
        "ci_2.5%": np.quantile(diff_boot, 0.025),
        "ci_97.5%": np.quantile(diff_boot, 0.975),
        "bootstrap_p_two_sided": p_two_sided,
    })

group_spectral_comparison = pd.DataFrame(group_comparison_rows)
print(f"Bootstrapped {n_pairs} matched cluster pairs for {BOOTSTRAP_REPS:,} replicates")
display(group_spectral_comparison)


Bootstrapped 98 matched cluster pairs for 10,000 replicates


,metric,high_mean,low_mean,high_minus_low,ci_2.5%,ci_97.5%,bootstrap_p_two_sided
0,delta_intrinsic_dim,-80.020408,-12.214286,-67.806122,-93.541582,-41.478316,0.000
1,delta_effective_rank,-109.744160,-9.517574,-100.226586,-128.795175,-72.381323,0.000
2,delta_var_first10,0.076905,0.026710,0.050195,0.021585,0.077829,0.001
3,delta_isotropy,-0.031009,-0.001460,-0.029549,-0.038041,-0.021656,0.000
4,cumulative_variance_area,0.022469,0.005146,0.017323,0.014121,0.020615,0.000


In [18]:
REPORT_COLUMNS = [
    "match_id", "group", "cluster", "rel_churn", "km_size",
    "pca_sample_n",
    "km_intrinsic_dim", "mfa_intrinsic_dim", "delta_intrinsic_dim",
    "km_effective_rank", "mfa_effective_rank", "delta_effective_rank",
    "km_var_first10", "mfa_var_first10", "delta_var_first10",
    "km_isotropy", "mfa_isotropy", "delta_isotropy",
    "cumulative_variance_area",
]
display(Markdown("**Matched clusters and their saved-spectrum changes:**"))
display(matched_spectral[REPORT_COLUMNS].sort_values(["match_id", "group"]))
display(Markdown("**High churn minus size-matched low churn:**"))
display(group_spectral_comparison)

if px is not None:
    plot_data = matched_spectral.melt(
        id_vars=["match_id", "group", "cluster"],
        value_vars=SPECTRAL_CHANGE_METRICS,
        var_name="metric", value_name="MFA - K-means",
    )
    fig = px.box(
        plot_data, x="group", y="MFA - K-means", color="group",
        facet_col="metric", facet_col_wrap=2, points="all",
        color_discrete_map={"high": "#d95f02", "low": "#7570b3"},
        category_orders={"group": ["low", "high"]},
        title="Saved-spectrum change: high churn vs size-matched low churn",
        template=PLOT_TEMPLATE,
    )
    fig.update_yaxes(matches=None, showticklabels=True, zeroline=True, zerolinecolor="#0b0b0b")
    fig.update_layout(height=950, width=1100, showlegend=False)
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    save_fig(fig, "high_vs_low_churn_spectral_change")
    fig.show()


**Matched clusters and their saved-spectrum changes:**

,match_id,group,cluster,rel_churn,km_size,pca_sample_n,km_intrinsic_dim,mfa_intrinsic_dim,delta_intrinsic_dim,km_effective_rank,mfa_effective_rank,delta_effective_rank,km_var_first10,mfa_var_first10,delta_var_first10,km_isotropy,mfa_isotropy,delta_isotropy,cumulative_variance_area
0,0,high,134,1.717116,49469,10000,253.0,234.0,-19.0,99.743052,73.056038,-26.687014,0.499219,0.574939,0.075720,0.011551,0.007334,-0.004217,0.003123
98,0,low,684,0.210190,49498,10000,286.0,283.0,-3.0,119.370734,115.085419,-4.285316,0.473809,0.483072,0.009264,0.014263,0.012420,-0.001843,0.001238
1,1,high,612,1.719305,21942,10000,381.0,127.0,-254.0,220.647529,50.421535,-170.225993,0.311810,0.631782,0.319972,0.023211,0.006077,-0.017133,0.038977
99,1,low,659,0.889881,22948,10000,231.0,324.0,93.0,94.567550,143.902921,49.335371,0.513024,0.431563,-0.081461,0.011191,0.013387,0.002196,0.013129
2,2,high,135,1.721522,19524,10000,205.0,190.0,-15.0,85.635443,91.373874,5.738431,0.520930,0.492462,-0.028468,0.009176,0.011236,0.002060,0.002620
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193,95,low,230,0.608911,32880,10000,193.0,212.0,19.0,69.704709,96.645181,26.940472,0.582259,0.487090,-0.095170,0.008195,0.010437,0.002242,0.004928
96,96,high,234,4.393344,16497,10000,593.0,426.0,-167.0,355.580508,215.888604,-139.691905,0.245070,0.340530,0.095459,0.018440,0.011933,-0.006507,0.028489
194,96,low,139,0.416040,32956,10000,351.0,296.0,-55.0,136.274150,108.312929,-27.961221,0.454399,0.496170,0.041770,0.015481,0.010703,-0.004778,0.006807
97,97,high,471,6.899450,14351,10000,41.0,403.0,362.0,21.856951,283.087990,261.231039,0.779366,0.264781,-0.514585,0.003917,0.022760,0.018843,0.064824


**High churn minus size-matched low churn:**

,metric,high_mean,low_mean,high_minus_low,ci_2.5%,ci_97.5%,bootstrap_p_two_sided
0,delta_intrinsic_dim,-80.020408,-12.214286,-67.806122,-93.541582,-41.478316,0.000
1,delta_effective_rank,-109.744160,-9.517574,-100.226586,-128.795175,-72.381323,0.000
2,delta_var_first10,0.076905,0.026710,0.050195,0.021585,0.077829,0.001
3,delta_isotropy,-0.031009,-0.001460,-0.029549,-0.038041,-0.021656,0.000
4,cumulative_variance_area,0.022469,0.005146,0.017323,0.014121,0.020615,0.000


In [19]:
display(Markdown("**Size-matching diagnostics:**"))
display(matches)
print(
    f"median matched size ratio: {np.exp(matches['log_size_gap'].median()):.3f}; "
    f"90th percentile: {np.exp(matches['log_size_gap'].quantile(0.9)):.3f}"
)


**Size-matching diagnostics:**

,match_id,high_cluster,low_cluster,high_rel_churn,low_rel_churn,high_km_size,low_km_size,pca_sample_n,log_size_gap
0,0,134,684,1.717116,0.210190,49469,49498,10000,0.000586
1,1,612,659,1.719305,0.889881,21942,22948,10000,0.044826
2,2,135,756,1.721522,0.762871,19524,19559,10000,0.001791
3,3,702,697,1.722905,0.036805,15464,16302,10000,0.052770
4,4,443,505,1.723100,0.749341,24164,24659,10000,0.020277
...,...,...,...,...,...,...,...,...,...
93,93,608,47,3.558487,0.632714,19218,32408,10000,0.522537
94,94,879,166,3.842758,0.394899,10964,32621,10000,1.090279
95,95,130,230,4.064526,0.608911,22487,32880,10000,0.379913
96,96,234,139,4.393344,0.416040,16497,32956,10000,0.691965


median matched size ratio: 1.007; 90th percentile: 1.694


### 5.2 Spectral Change Across All Clusters

Section 5 compared only the top-10% churn clusters against size-matched
low-churn controls. Here the same saved-spectrum deltas (`MFA - K-means`) are
reported for **every** equal-sample cluster in `spectral_by_cluster`, to answer
whether training raises or lowers intrinsic dimension overall — not just for
the extreme movers.

> **Why this disagrees with `kmeans_mfa_comparison.ipynb` section 4.1.** That
> notebook loads its K-means IDs from
> `dalg-cache/output/experiments/centroids_1000_05/intrinsic_dims.pt`, which was
> computed with `max_samples=2000`, while its MFA IDs use `max_samples=10000`.
> The ID at a fixed variance threshold grows strongly with PCA sample count
> (this same K-means partition scores mean ID ≈ 247 at 2k samples vs ≈ 308 at
> 10k), so the apparent ΔID ≈ +8 there is a sample-size artifact, not a
> geometric effect. With matched 10k-sample artifacts (used here), ΔID is
> clearly negative.

In [20]:
SPECTRAL_SEED_ALL = 20260716
rng_all = np.random.default_rng(SPECTRAL_SEED_ALL)
boot_idx_all = rng_all.integers(
    0, len(spectral_by_cluster), size=(BOOTSTRAP_REPS, len(spectral_by_cluster))
)

all_cluster_rows = []
for metric in SPECTRAL_CHANGE_METRICS:
    values = spectral_by_cluster[metric].to_numpy(dtype=np.float64)
    boot_means = values[boot_idx_all].mean(axis=1)
    all_cluster_rows.append({
        "metric": metric,
        "mean": values.mean(),
        "median": np.median(values),
        "frac_negative": float((values < 0).mean()),
        "frac_positive": float((values > 0).mean()),
        "ci_2.5%": np.quantile(boot_means, 0.025),
        "ci_97.5%": np.quantile(boot_means, 0.975),
    })
all_cluster_spectral = pd.DataFrame(all_cluster_rows)
print(f"All equal-sample clusters: {len(spectral_by_cluster)}/{K}")
display(all_cluster_spectral)

spectral_by_cluster["churn_quintile"] = pd.qcut(
    spectral_by_cluster["rel_churn"], 5, labels=[f"Q{i}" for i in range(1, 6)]
)
quintile_table = spectral_by_cluster.groupby("churn_quintile", observed=True).agg(
    n=("cluster", "size"),
    rel_churn_median=("rel_churn", "median"),
    delta_id_mean=("delta_intrinsic_dim", "mean"),
    delta_id_median=("delta_intrinsic_dim", "median"),
    frac_id_negative=("delta_intrinsic_dim", lambda x: (x < 0).mean()),
    delta_effective_rank_mean=("delta_effective_rank", "mean"),
)
display(Markdown("**ΔID by relative-churn quintile (Q1 = lowest churn):**"))
display(quintile_table)

from scipy.stats import spearmanr

rho_all, p_all = spearmanr(
    spectral_by_cluster["rel_churn"], spectral_by_cluster["delta_intrinsic_dim"]
)
print(f"Spearman(rel_churn, delta_intrinsic_dim): rho={rho_all:.3f}, p={p_all:.2e}")

if px is not None:
    fig = px.histogram(
        spectral_by_cluster,
        x="delta_intrinsic_dim",
        nbins=60,
        title="ΔID = ID(mfa) − ID(kmeans), all equal-sample clusters (matched 10k-sample artifacts)",
        color_discrete_sequence=[COLOR_NEUTRAL],
        template=PLOT_TEMPLATE,
    )
    fig.add_vline(x=0, line_dash="dash", line_color="#0b0b0b")
    fig.update_layout(xaxis_title="ΔID (trained − init)", yaxis_title="clusters", bargap=0.05)
    save_fig(fig, "delta_id_hist_all_clusters")
    fig.show()

    fig = px.scatter(
        spectral_by_cluster,
        x="rel_churn",
        y="delta_intrinsic_dim",
        hover_data=["cluster", "km_size", "km_intrinsic_dim", "mfa_intrinsic_dim"],
        title="ΔID vs relative churn, all equal-sample clusters",
        color_discrete_sequence=[COLOR_NEUTRAL],
        template=PLOT_TEMPLATE,
        opacity=0.55,
    )
    fig.add_hline(y=0, line_dash="dash", line_color="#0b0b0b")
    fig.update_layout(xaxis_title="relative churn (left + joined) / km size", yaxis_title="ΔID")
    save_fig(fig, "delta_id_vs_rel_churn_all_clusters")
    fig.show()

All equal-sample clusters: 979/1000


,metric,mean,median,frac_negative,frac_positive,ci_2.5%,ci_97.5%
0,delta_intrinsic_dim,-52.989785,-29.000000,0.718080,0.273749,-58.769178,-47.274617
1,delta_effective_rank,-59.733277,-28.081057,0.792646,0.207354,-65.711242,-53.969794
2,delta_var_first10,0.079755,0.058174,0.195097,0.804903,0.073292,0.086307
3,delta_isotropy,-0.013325,-0.004135,0.793667,0.206333,-0.014892,-0.011808
4,cumulative_variance_area,0.012138,0.006320,0.000000,1.000000,0.011261,0.013026


**ΔID by relative-churn quintile (Q1 = lowest churn):**

,n,rel_churn_median,delta_id_mean,delta_id_median,frac_id_negative,delta_effective_rank_mean
churn_quintile,,,,,,
Q1,196,0.228735,-8.724490,-7.0,0.637755,-8.476289
Q2,196,0.576913,-27.117347,-22.0,0.663265,-21.352766
Q3,195,0.894624,-47.369231,-32.0,0.738462,-48.197352
Q4,196,1.198712,-95.505102,-71.0,0.795918,-109.235819
Q5,196,1.716923,-86.204082,-92.0,0.755102,-111.345304


Spearman(rel_churn, delta_intrinsic_dim): rho=-0.351, p=1.05e-29


## 6. Section 4.6 Outliers vs Size-Matched Controls

Repeat the saved-spectrum analysis using the outliers identified in Section 4.6
instead of selecting clusters by relative churn. An **outlier cluster** is any
unique cluster appearing in either `distance_outliers` or `id_gap_outliers`, in
any of the `give`, `take`, or `give+take` strata. The stricter
`overlap_outliers` table contains only one unique cluster (two cluster-stratum
rows), which is too small for a group comparison.

Controls exclude every Section 4.6 outlier and are matched without replacement by
K-means cluster size and saved PCA sample count. The spectral changes and
matched-pair bootstrap are otherwise identical to Section 5.


In [21]:
outlier_events = pd.concat(
    [
        distance_outliers[["cluster", "stratum"]].assign(
            outlier_source="centroid_distance"
        ),
        id_gap_outliers[["cluster", "stratum"]].assign(
            outlier_source="intrinsic_dim_gap"
        ),
    ],
    ignore_index=True,
).drop_duplicates()
outlier_catalog = (
    outlier_events.groupby("cluster")
    .agg(
        outlier_sources=("outlier_source", lambda x: ", ".join(sorted(set(x)))),
        outlier_strata=("stratum", lambda x: ", ".join(sorted(set(x)))),
    )
    .reset_index()
)
all_outlier_ids = set(outlier_catalog["cluster"].astype(int))
outlier_selected = spectral_by_cluster.merge(outlier_catalog, on="cluster", how="inner")
control_pool = spectral_by_cluster[
    ~spectral_by_cluster["cluster"].isin(all_outlier_ids)
].copy()
assert len(outlier_selected) > 1
assert len(control_pool) >= len(outlier_selected)

outlier_cost = np.abs(
    np.log1p(outlier_selected["km_size"].to_numpy())[:, None]
    - np.log1p(control_pool["km_size"].to_numpy())[None, :]
)
outlier_cost += 1e6 * (
    outlier_selected["pca_sample_n"].to_numpy()[:, None]
    != control_pool["pca_sample_n"].to_numpy()[None, :]
)
outlier_row, control_row = linear_sum_assignment(outlier_cost)
outlier_selected = outlier_selected.iloc[outlier_row].copy().reset_index(drop=True)
outlier_controls = control_pool.iloc[control_row].copy().reset_index(drop=True)
assert np.array_equal(outlier_selected["pca_sample_n"], outlier_controls["pca_sample_n"])

n_outlier_pairs = len(outlier_selected)
outlier_matches = pd.DataFrame({
    "match_id": np.arange(n_outlier_pairs),
    "outlier_cluster": outlier_selected["cluster"],
    "control_cluster": outlier_controls["cluster"],
    "outlier_sources": outlier_selected["outlier_sources"],
    "outlier_strata": outlier_selected["outlier_strata"],
    "outlier_rel_churn": outlier_selected["rel_churn"],
    "control_rel_churn": outlier_controls["rel_churn"],
    "outlier_km_size": outlier_selected["km_size"],
    "control_km_size": outlier_controls["km_size"],
    "pca_sample_n": outlier_selected["pca_sample_n"],
})
outlier_matches["log_size_gap"] = np.abs(
    np.log1p(outlier_matches["outlier_km_size"])
    - np.log1p(outlier_matches["control_km_size"])
)
outlier_selected["match_id"], outlier_selected["group"] = (
    outlier_matches["match_id"], "outlier"
)
outlier_controls["match_id"], outlier_controls["group"] = (
    outlier_matches["match_id"], "control"
)
outlier_controls["outlier_sources"] = "not an outlier"
outlier_controls["outlier_strata"] = ""
matched_outlier_spectral = pd.concat(
    [outlier_selected, outlier_controls], ignore_index=True
)

print(
    f"Section 4.6 unique outlier clusters: {len(all_outlier_ids)}; "
    f"with equal-sample spectra: {n_outlier_pairs}; excluded: "
    f"{len(all_outlier_ids) - n_outlier_pairs}"
)
display(outlier_matches.describe(percentiles=[0.1, 0.5, 0.9]))


Section 4.6 unique outlier clusters: 31; with equal-sample spectra: 19; excluded: 12


,match_id,outlier_cluster,control_cluster,outlier_rel_churn,control_rel_churn,outlier_km_size,control_km_size,pca_sample_n,log_size_gap
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.0,19.000000
mean,9.000000,422.052632,505.736842,1.181761,1.172594,51654.000000,51643.842105,10000.0,0.003163
std,5.627314,286.787857,361.879330,1.550599,0.774620,53735.870942,53703.126520,0.0,0.002855
min,0.000000,7.000000,29.000000,0.006210,0.193720,12226.000000,12140.000000,10000.0,0.000085
10%,1.800000,62.800000,88.800000,0.060541,0.441066,14156.000000,14204.400000,10000.0,0.000442
50%,9.000000,471.000000,532.000000,0.945528,1.053957,25657.000000,25613.000000,10000.0,0.002436
90%,16.200000,799.600000,956.400000,1.970582,2.097239,143016.600000,142875.600000,10000.0,0.007166
max,18.000000,897.000000,996.000000,6.899450,2.926024,184863.000000,184724.000000,10000.0,0.008733


In [22]:
OUTLIER_BOOTSTRAP_REPS = 10_000
OUTLIER_SPECTRAL_SEED = 20260716
outlier_rng = np.random.default_rng(OUTLIER_SPECTRAL_SEED)

outlier_values = outlier_selected[SPECTRAL_CHANGE_METRICS].to_numpy(dtype=np.float64)
control_values = outlier_controls[SPECTRAL_CHANGE_METRICS].to_numpy(dtype=np.float64)
outlier_bootstrap_indices = outlier_rng.integers(
    0, n_outlier_pairs, size=(OUTLIER_BOOTSTRAP_REPS, n_outlier_pairs)
)

outlier_comparison_rows = []
for metric_i, metric in enumerate(SPECTRAL_CHANGE_METRICS):
    outlier_boot = outlier_values[outlier_bootstrap_indices, metric_i].mean(axis=1)
    control_boot = control_values[outlier_bootstrap_indices, metric_i].mean(axis=1)
    diff_boot = outlier_boot - control_boot
    p_two_sided = min(
        1.0, 2 * min(np.mean(diff_boot <= 0), np.mean(diff_boot >= 0))
    )
    outlier_comparison_rows.append({
        "metric": metric,
        "outlier_mean": outlier_values[:, metric_i].mean(),
        "control_mean": control_values[:, metric_i].mean(),
        "outlier_minus_control": (
            outlier_values[:, metric_i].mean() - control_values[:, metric_i].mean()
        ),
        "ci_2.5%": np.quantile(diff_boot, 0.025),
        "ci_97.5%": np.quantile(diff_boot, 0.975),
        "bootstrap_p_two_sided": p_two_sided,
    })

outlier_spectral_comparison = pd.DataFrame(outlier_comparison_rows)
print(
    f"Bootstrapped {n_outlier_pairs} outlier/control pairs for "
    f"{OUTLIER_BOOTSTRAP_REPS:,} replicates"
)
display(outlier_spectral_comparison)


Bootstrapped 19 outlier/control pairs for 10,000 replicates


,metric,outlier_mean,control_mean,outlier_minus_control,ci_2.5%,ci_97.5%,bootstrap_p_two_sided
0,delta_intrinsic_dim,-37.421053,-35.368421,-2.052632,-77.475000,72.947368,0.9594
1,delta_effective_rank,-82.921215,-56.706889,-26.214326,-120.862813,61.648802,0.5944
2,delta_var_first10,0.020948,0.081985,-0.061037,-0.141598,0.016318,0.1186
3,delta_isotropy,-0.025937,-0.012945,-0.012992,-0.038932,0.010366,0.2976
4,cumulative_variance_area,0.019800,0.010589,0.009211,-0.002600,0.021717,0.1328


In [23]:
OUTLIER_REPORT_COLUMNS = [
    "match_id", "group", "cluster", "outlier_sources", "outlier_strata",
    "rel_churn", "km_size", "pca_sample_n",
    "km_intrinsic_dim", "mfa_intrinsic_dim", "delta_intrinsic_dim",
    "km_effective_rank", "mfa_effective_rank", "delta_effective_rank",
    "km_var_first10", "mfa_var_first10", "delta_var_first10",
    "km_isotropy", "mfa_isotropy", "delta_isotropy",
    "cumulative_variance_area",
]
display(Markdown("**Section 4.6 outliers and their size-matched controls:**"))
display(
    matched_outlier_spectral[OUTLIER_REPORT_COLUMNS]
    .sort_values(["match_id", "group"])
)
display(Markdown("**Outliers minus size-matched controls:**"))
display(outlier_spectral_comparison)
print(
    f"median matched size ratio: "
    f"{np.exp(outlier_matches['log_size_gap'].median()):.3f}; "
    f"90th percentile: {np.exp(outlier_matches['log_size_gap'].quantile(0.9)):.3f}"
)

if px is not None:
    outlier_plot_data = matched_outlier_spectral.melt(
        id_vars=["match_id", "group", "cluster"],
        value_vars=SPECTRAL_CHANGE_METRICS,
        var_name="metric", value_name="MFA - K-means",
    )
    fig = px.box(
        outlier_plot_data, x="group", y="MFA - K-means", color="group",
        facet_col="metric", facet_col_wrap=2, points="all",
        color_discrete_map={"outlier": "#d95f02", "control": "#7570b3"},
        category_orders={"group": ["control", "outlier"]},
        title="Saved-spectrum change: Section 4.6 outliers vs size-matched controls",
        template=PLOT_TEMPLATE,
    )
    fig.update_yaxes(
        matches=None, showticklabels=True, zeroline=True, zerolinecolor="#0b0b0b"
    )
    fig.update_layout(height=950, width=1100, showlegend=False)
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    save_fig(fig, "section46_outliers_spectral_change")
    fig.show()


**Section 4.6 outliers and their size-matched controls:**

,match_id,group,cluster,outlier_sources,outlier_strata,rel_churn,km_size,pca_sample_n,km_intrinsic_dim,mfa_intrinsic_dim,...,km_effective_rank,mfa_effective_rank,delta_effective_rank,km_var_first10,mfa_var_first10,delta_var_first10,km_isotropy,mfa_isotropy,delta_isotropy,cumulative_variance_area
19,0,control,952,not an outlier,,1.680207,18709,10000,81.0,187.0,...,44.620347,107.150434,62.530088,0.639490,0.453998,-0.185491,0.006590,0.014639,0.008049,0.013857
0,0,outlier,7,centroid_distance,take,1.130849,18785,10000,176.0,253.0,...,62.861034,101.118766,38.257731,0.593096,0.487491,-0.105605,0.007091,0.009108,0.002017,0.010325
20,1,control,112,not an outlier,,1.472652,12140,10000,75.0,74.0,...,51.533738,37.125298,-14.408440,0.586187,0.676103,0.089916,0.008903,0.005157,-0.003746,0.002973
1,1,outlier,62,centroid_distance,"give, give+take, take",1.674628,12226,10000,597.0,319.0,...,312.915769,108.858989,-204.056780,0.289591,0.457017,0.167425,0.018982,0.006477,-0.012505,0.046591
21,2,control,358,not an outlier,,2.926024,16208,10000,585.0,364.0,...,475.779638,212.173870,-263.605768,0.171976,0.343369,0.171393,0.080955,0.021800,-0.059156,0.040795
2,2,outlier,63,intrinsic_dim_gap,"give+take, take",0.945528,16302,10000,13.0,58.0,...,7.486197,24.793805,17.307608,0.891166,0.758731,-0.132435,0.001301,0.003474,0.002173,0.009011
22,3,control,210,not an outlier,,1.053957,164706,10000,305.0,305.0,...,203.218924,111.048048,-92.170876,0.290413,0.516622,0.226209,0.036734,0.012842,-0.023893,0.005746
3,3,outlier,78,intrinsic_dim_gap,take,0.279919,164619,10000,402.0,392.0,...,192.733882,161.072390,-31.661492,0.377095,0.420630,0.043535,0.023905,0.017029,-0.006876,0.002622
23,4,control,116,not an outlier,,0.306918,23635,10000,156.0,131.0,...,54.487979,37.855816,-16.632163,0.611908,0.684244,0.072336,0.005910,0.004187,-0.001723,0.003047
4,4,outlier,104,intrinsic_dim_gap,give,1.005797,23633,10000,82.0,115.0,...,32.152941,51.964283,19.811342,0.702912,0.622370,-0.080542,0.004360,0.006944,0.002584,0.005181


**Outliers minus size-matched controls:**

,metric,outlier_mean,control_mean,outlier_minus_control,ci_2.5%,ci_97.5%,bootstrap_p_two_sided
0,delta_intrinsic_dim,-37.421053,-35.368421,-2.052632,-77.475000,72.947368,0.9594
1,delta_effective_rank,-82.921215,-56.706889,-26.214326,-120.862813,61.648802,0.5944
2,delta_var_first10,0.020948,0.081985,-0.061037,-0.141598,0.016318,0.1186
3,delta_isotropy,-0.025937,-0.012945,-0.012992,-0.038932,0.010366,0.2976
4,cumulative_variance_area,0.019800,0.010589,0.009211,-0.002600,0.021717,0.1328


median matched size ratio: 1.002; 90th percentile: 1.007


## 7. Summary

Compact table of the headline numbers.


In [24]:
summary = pd.DataFrame(
    [
        {"metric": "same-id agreement", "value": same_id_agreement},
        {"metric": "tokens changing cluster", "value": int(N - shared.sum())},
        {"metric": "median per-cluster churn", "value": float(per_cluster["churn"].median())},
        {"metric": "median relative churn", "value": float(per_cluster["rel_churn"].median())},
        {"metric": "clusters with rel_churn > 1", "value": int((per_cluster["rel_churn"] > 1).sum())},
        {"metric": "clusters that grew (net_delta > 0)", "value": int((per_cluster["net_delta"] > 0).sum())},
        {"metric": "clusters that shrank (net_delta < 0)", "value": int((per_cluster["net_delta"] < 0).sum())},
        {"metric": "max churn cluster", "value": int(per_cluster.loc[per_cluster["churn"].idxmax(), "cluster"])},
    ]
)
display(summary)


,metric,value
0,same-id agreement,5.730124e-01
1,tokens changing cluster,3.146384e+07
2,median per-cluster churn,4.504900e+04
3,median relative churn,9.011186e-01
4,clusters with rel_churn > 1,4.180000e+02
5,clusters that grew (net_delta > 0),5.930000e+02
6,clusters that shrank (net_delta < 0),4.070000e+02
7,max churn cluster,3.830000e+02
